<a href="https://colab.research.google.com/github/kylampiima/FYP/blob/main/FYP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pydub
!pip install transformers datasets evaluate soundfile librosa accelerate>=0.21.0
!pip install -q transformers
!pip install gtts
!pip install datasets

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
# -*- coding: utf-8 -*-
"""
WIP_Data_generation.py

This script processes audio data from the speech_commands dataset,
applies a stutter effect, and prepares the data for training a model
using the Wav2Vec2 architecture.
"""

import soundfile as sf
import librosa
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Tokenizer, AutoFeatureExtractor, Trainer, TrainingArguments, AutoModelForAudioClassification
from pydub import AudioSegment
from datasets import load_dataset, Dataset, Audio, ClassLabel, Value, Features
import numpy as np
from sklearn.model_selection import train_test_split
import evaluate


    # Load the dataset with trust_remote_code set to True
ds = load_dataset("speech_commands", "v0.02", split="train", trust_remote_code=True)

modified_dataset = []

# Process existing dataset samples
for i in range(40000):
    if i % 10 == 0:
        sample = ds[i]
        audio_data = np.array(sample["audio"]["array"])
        sampling_rate = sample["audio"]["sampling_rate"]

        # Convert NumPy array to an audio segment
        audio = AudioSegment(
            (audio_data * 32767).astype(np.int16).tobytes(),
            frame_rate=sampling_rate,
            sample_width=2,
            channels=1
        )

        # Extract the first 200ms for the stutter effect
        first_sound = audio[:200]
        first_sound = first_sound.set_frame_rate(sampling_rate).set_sample_width(2).set_channels(1)

        # Repeat the first sound a few times
        stutter_effect = first_sound * 3

        # Merge the repeated sound with the original audio
        output_audio = stutter_effect + audio

        # Export the modified audio
        output_filename = f"/wav/output_sample_{i}.wav"
        output_audio.export(output_filename, format="wav")

        # Create a modified sample entry for the processed audio
        modified_sample = {
            "file": output_filename,
            "audio": output_filename,
            "label": sample["label"],
            "is_unknown": sample["is_unknown"],
            "speaker_id": sample["speaker_id"],
            "utterance_id": sample["utterance_id"]
        }

        modified_dataset.append(modified_sample)

print("Audio processing complete. Check output_sample.wav")

# Define class labels
class_labels = ClassLabel(names=[
    'yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off',
    'stop', 'go', 'zero', 'one', 'two', 'three', 'four',
    'five', 'six', 'seven', 'eight', 'nine', 'bed', 'bird',
    'cat', 'dog', 'happy', 'house', 'marvin', 'sheila',
    'tree', 'wow', 'backward', 'forward', 'follow', 'learn',
    'visual', '_silence_','Sometimes I feel a little shy.',
    'Today is a great day.',
    'I really love to play the guitar.',
    'Wait for me at the corner of the street.',
    'Thank you for your help today.',
    'Sometimes I just want to relax and read.',
    'Could you please pass the salt?',
    'Here comes the sun after the rain.',
    'Maybe we can go to the movies later.',
    'Friends are important in our lives.',
    'You should try that new restaurant downtown.',
    'Little things can make a big difference.',
    "Hello my name is tester"
])

#the features for the dataset
features = Features({
    'file': Value(dtype='string'),
    'audio': Audio(sampling_rate=16000, mono=True, decode=True),
    'label': class_labels,
    'is_unknown': Value(dtype='bool'),
    'speaker_id': Value(dtype='string'),
    'utterance_id': Value(dtype='int8')
})

#new dataset from the modified dataset
new_train_ds = Dataset.from_list(modified_dataset)
new_train_ds = new_train_ds.cast(features)

print("Original dataset size:", len(new_train_ds))

#train and test sets
train_test_split = new_train_ds.train_test_split(test_size=0.2)
train_ds = train_test_split['train']
test_ds = train_test_split['test']

#tokenizer and model
tokenizer = Wav2Vec2Tokenizer.from_pretrained("facebook/wav2vec2-base-960h")
model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-large-960h")

# Removing unnecessary columns
train_ds = train_ds.remove_columns(["is_unknown", "speaker_id", "utterance_id"])

# Preparing label mappings
labels = train_ds.features["label"].names
label2id = {label: str(i) for i, label in enumerate(labels)}
id2label = {str(i): label for i, label in enumerate(labels)}

#feature extractor
feature_extractor = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base")
train_ds = train_ds.cast_column("audio", Audio(sampling_rate=16_000))

# Preprocess function
def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays, sampling_rate=feature_extractor.sampling_rate, max_length=16000, truncation=True
    )
    return inputs

# Encode datasets
encoded_train_ds = train_ds.map(preprocess_function, remove_columns="audio", batched=True)
encoded_eval_ds = test_ds.map(preprocess_function, remove_columns="audio", batched=True)

# Load accuracy metric
accuracy = evaluate.load("accuracy")
print(accuracy)

# Compute metrics function
def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=eval_pred.label_ids)

# Prepare model for audio classification
num_labels = len(id2label)
model = AutoModelForAudioClassification.from_pretrained(
    "facebook/wav2vec2-base", num_labels=num_labels, label2id=label2id, id2label=id2label
)

# Set training arguments
training_args = TrainingArguments(
    output_dir="my_fyp_test12",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=32,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=32,
    num_train_epochs=100,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_train_ds,
    eval_dataset=encoded_eval_ds,
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics,
)

# Train the model
trainer.train()

# Save the model and feature extractor
model.save_pretrained("/content/drive/my_fyp_test11")
feature_extractor.save_pretrained("/content/drive/my_fyp_test11")



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/12.1k [00:00<?, ?B/s]

speech_commands.py:   0%|          | 0.00/7.31k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/84848 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/9982 [00:00<?, ? examples/s]

...

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mpiimakyla (mpiimakyla-mtu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy
1,3.886200,3.863861,0.067500
2,3.805300,3.772241,0.087500
3,3.683600,3.611185,0.095000
4,3.538800,3.495600,0.092500
5,3.453800,3.396635,0.080000
6,3.322400,3.288057,0.103750
7,3.240400,3.176487,0.118750
8,3.112700,3.065070,0.136250
9,3.017200,2.948933,0.200000
10,2.901800,2.856936,0.231250


['/content/drive/my_fyp_test11/preprocessor_config.json']

In [ ]:

predictions = trainer.predict(encoded_eval_ds)

predicted_labels = np.argmax(predictions.predictions, axis=1)

true_labels = predictions.label_ids

cm = confusion_matrix(true_labels, predicted_labels)



In [ ]:
cm

array([[12,  0,  0,  2,  0,  1,  0,  6,  0,  0,  1,  0,  1,  0,  1,  1,
         0,  0,  2],
       [ 1, 35,  0,  4,  0,  1,  0,  3,  3,  2,  3,  0,  4,  1,  1,  1,
         0,  0,  1],
       [ 1,  3, 38,  3,  1,  1,  5,  2,  1,  1,  0,  0,  2,  1,  1,  1,
         0,  0,  2],
       [ 0,  4,  4, 30,  3,  2,  1,  0,  3,  2,  2,  0,  2,  3,  0,  2,
         0,  0,  3],
       [ 0,  2,  2,  0, 33,  6,  2,  1,  4,  2,  3,  1,  1,  1,  2,  1,
         0,  0,  1],
       [ 0,  2,  4,  1,  7, 35,  1,  2,  2,  2,  3,  0,  1,  1,  1,  4,
         0,  0,  1],
       [ 0,  3,  3,  3,  2,  3, 31,  1,  7,  1,  1,  0,  1,  1,  1,  3,
         0,  0,  0],
       [ 4,  5,  2,  3,  4,  3,  0, 26,  3,  1,  0,  0,  3,  1,  1,  5,
         0,  0,  5],
       [ 0,  4,  0,  3,  0,  0,  1,  2,  9,  5,  0,  1,  3,  0,  1,  6,
         0,  0,  0],
       [ 0,  4,  0,  1,  1,  1,  1,  1,  3, 11,  1,  0,  1,  0,  0,  3,
         0,  0,  4],
       [ 0,  3,  2,  2,  1,  0,  1,  1,  3,  1, 15,  1,  5,  1,  0,  1

In [ ]:
plt.scatter(true_labels[:,0], predicted_labels[:,1],c=true_labels)

In [ ]:
class_names = list(id2label.values())
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.show()